# Lisa Dlima — personal site

A minimal single-page personal site for Lisa Dlima. The whole page is a Monte-Carlo localization ("particle filter") world: the robot drives around landmarks, and a cloud of particles converges on its position. Lisa's info card floats in the center.

Drive with the arrow keys on desktop, or tilt controls on mobile.

## Setup

FastHTML plus the semantic custom tags we use to build the card and world.

In [ ]:
from fasthtml.common import *
from fasthtml.components import Slam_World, Info_Card, Role_Title, Hint_Bar, Profile_Pic, Bio_Text, Exp_List, Exp_Item, Edu_Line, Pub_Section, Pub_Head, Pub_Item, Card_Links, Tilt_Btn, Lesson_Section, Lesson_Head, Lesson_Item

## Styling

All CSS in one place, targeting our semantic tag names directly. The card caps its height and scrolls internally so it never overflows on scaled displays, and the mobile block swaps in a touch-friendly layout plus the tilt-enable button.

In [ ]:
css = """
*, *::before, *::after { box-sizing: border-box; }
body { margin: 0; font-family: system-ui, sans-serif; background: #0d0d17; color: #e8e8f0; overflow: hidden; }

slam-world { position: fixed; inset: 0; }
slam-world canvas { display: block; width: 100%; height: 100%; }

info-card { position: fixed; top: 50%; left: 50%; transform: translate(-50%,-50%); z-index: 10;
  background: rgba(20,20,35,0.85); backdrop-filter: blur(8px); padding: 2.5rem 3rem; border-radius: 1rem;
  border: 1px solid rgba(255,255,255,0.08); text-align: center; max-width: 90vw; max-height: 90vh; overflow-y: auto; }
info-card h1 { margin: 0 0 0.25rem; font-size: 2rem; }
info-card role-title { display: block; color: #8b8bd0; font-size: 1rem; margin-bottom: 1rem; }
info-card a { color: #6ba3ff; text-decoration: none; }

profile-pic { display:block; width:96px; height:96px; margin:0 auto 1rem; border-radius:50%; overflow:hidden; background:#2a2a40; border:1px solid rgba(255,255,255,0.1); }
profile-pic img { width:100%; height:100%; object-fit:cover; display:block; }
bio-text { display:block; color:#c8c8d8; font-size:0.95rem; line-height:1.5; margin:0 auto 1rem; max-width:44ch; }
exp-list { display:block; margin-bottom:1rem; }
exp-item { display:block; color:#a8a8c0; font-size:0.9rem; padding:0.15rem 0; }
edu-line { display:block; color:#8b8bd0; font-size:0.9rem; margin-bottom:1rem; }

pub-section { display:block; margin-bottom:1rem; }
pub-head { display:block; color:#6a6a85; font-size:0.75rem; letter-spacing:0.08em; text-transform:uppercase; margin-bottom:0.4rem; }
pub-item { display:block; color:#a8a8c0; font-size:0.85rem; line-height:1.4; padding:0.15rem 0; }

lesson-section { display:block; margin-bottom:1rem; }
lesson-head { display:block; color:#6a6a85; font-size:0.75rem; letter-spacing:0.08em; text-transform:uppercase; margin-bottom:0.4rem; }
lesson-item { display:block; font-size:0.9rem; line-height:1.4; padding:0.15rem 0; }
card-links { display:flex; gap:1.5rem; justify-content:center; }
card-links a { display:inline-flex; transition:color 0.15s; }
card-links a:hover { color:#8b8bd0; }

hint-bar { position: fixed; bottom: 1rem; left: 50%; transform: translateX(-50%); z-index: 10; font-size: 0.85rem; color: #6a6a85; }
tilt-btn { display:none; }

@media (max-width: 600px) {
  info-card { width:94vw; max-width:94vw; max-height:92vh; padding:1.5rem 1.25rem; }
  info-card h1 { font-size:1.6rem; }
  bio-text { font-size:0.9rem; }
  hint-bar { display:none; }
  tilt-btn { display:block; position:fixed; bottom:1.25rem; left:50%; transform:translateX(-50%); z-index:20;
    background:#6ba3ff; color:#0d0d17; font-size:0.9rem; font-weight:600; padding:0.7rem 1.4rem; border-radius:2rem; border:none; }
}
"""

## The particle filter

The interactive world, written in JS and served as a `<script>`. Key pieces:

- **`resize`** sizes the canvas for the device pixel ratio (sharp on retina), scatters landmarks (avoiding the center card), and places the robot at a random spot.
- **`sense`** returns each landmark's distance, with a shared per-tick dropout `mask` so a flickering landmark affects the robot and every particle equally.
- **`update`** weights particles by how well their sensed distances match the robot's, then low-variance resamples with jitter.
- **`move`** applies motion (with noise) to robot and particles. When idle, particles slowly diffuse — the robot "loses certainty".
- Controls: arrow keys, plus optional device-tilt (roll to turn, pitch past ~45° to drive).

In [ ]:
demo_js = """
const cv = document.getElementById('world'), ctx = cv.getContext('2d');
let W, H, obstacles = [], particles = [];
const robot = {x:0, y:0, th:0}, N = 600;

function scatter() {
  particles = [];
  for (let i=0; i<N; i++) particles.push({x:Math.random()*W, y:Math.random()*H, th:Math.random()*7, w:1/N});
}

function resize() {
  const dpr = devicePixelRatio || 1;
  W = innerWidth; H = innerHeight;
  cv.width = W*dpr; cv.height = H*dpr;
  cv.style.width = W+'px'; cv.style.height = H+'px';
  ctx.setTransform(dpr,0,0,dpr,0,0);
  do { robot.x = Math.random()*W; robot.y = Math.random()*H; } while (Math.hypot(robot.x-W/2, robot.y-H/2) < 200);
  robot.th = Math.random()*7;
  obstacles = [];
  for (let i=0; i<10; i++) {
    const x = Math.random()*W, y = Math.random()*H;
    if (Math.hypot(x-W/2, y-H/2) < 180) continue;
    obstacles.push({x, y});
  }
  scatter();
}
addEventListener('resize', resize); resize();

function sense(x, y, mask) {
  const d = new Array(obstacles.length);
  for (let i=0; i<obstacles.length; i++) d[i] = mask[i] ? 9999 : Math.hypot(x-obstacles[i].x, y-obstacles[i].y);
  return d;
}

function gauss() { return (Math.random()+Math.random()+Math.random()+Math.random()-2)/2; }

function update() {
    const mask = obstacles.map(o => Math.random()<0.2);
    const z = sense(robot.x, robot.y, mask);
  let sum = 0;
  for (const p of particles) {
    const zp = sense(p.x, p.y, mask);
    let e = 0;
    for (let i=0; i<z.length; i++) e += (z[i]-zp[i])**2;
    p.w = Math.exp(-e/(2*60*60)); sum += p.w;
  }
  for (const p of particles) p.w /= (sum||1);
  const next = [], step = 1/N; let r = Math.random()*step, c = particles[0].w, i = 0;
  for (let m=0; m<N; m++) {
    const u = r + m*step;
    while (u > c && i < N-1) { i++; c += particles[i].w; }
    const s = particles[i];
    next.push({x:s.x+gauss()*8, y:s.y+gauss()*8, th:s.th, w:step});
  }
  particles = next;
}

function move(dx, dy, dth) {
  robot.th += dth;
  robot.x += dx; robot.y += dy;
  for (const p of particles) { p.th += dth; p.x += dx+gauss()*2; p.y += dy+gauss()*2; }
  update();
}

function drawRobot() {
  ctx.save(); ctx.translate(robot.x, robot.y); ctx.rotate(robot.th);
  ctx.fillStyle = '#6ba3ff';
  ctx.beginPath(); ctx.moveTo(14,0); ctx.lineTo(-10,-9); ctx.lineTo(-10,9); ctx.closePath(); ctx.fill();
  ctx.restore();
}

function draw() {
  ctx.clearRect(0,0,W,H);
  ctx.fillStyle = 'rgba(120,200,120,0.5)';
  for (const p of particles) { ctx.beginPath(); ctx.arc(p.x,p.y,1.5,0,7); ctx.fill(); }
  ctx.fillStyle = '#2a2a40';
  for (const o of obstacles) { ctx.beginPath(); ctx.arc(o.x,o.y,5,0,7); ctx.fill(); }
  drawRobot();
  requestAnimationFrame(draw);
}
draw();

const keys = {};
addEventListener('keydown', e => { keys[e.key]=1; if(e.key.startsWith('Arrow')) e.preventDefault(); });
addEventListener('keyup', e => keys[e.key]=0);

let tiltOn = false, tiltX = 0, tiltY = 0;
function onTilt(e) { tiltX = e.gamma||0; tiltY = e.beta||0; }
function enableTilt() {
  if (typeof DeviceOrientationEvent.requestPermission === 'function') DeviceOrientationEvent.requestPermission().then(s => { if (s==='granted') { addEventListener('deviceorientation', onTilt); tiltOn=true; } });
  else { addEventListener('deviceorientation', onTilt); tiltOn=true; }
}
window.enableTilt = enableTilt;
setInterval(() => {
  let dx = 0, dy = 0, dth = 0;
  if (keys.ArrowLeft) dth -= 0.08;
  if (keys.ArrowRight) dth += 0.08;
  if (keys.ArrowUp) { dx = Math.cos(robot.th)*4; dy = Math.sin(robot.th)*4; }
  if (keys.ArrowDown) { dx = -Math.cos(robot.th)*4; dy = -Math.sin(robot.th)*4; }
    if (tiltOn) {
      if (Math.abs(tiltX) > 8) dth += tiltX/600;
      if (Math.abs(tiltY-45) > 12) { const s = (tiltY-45)/60; dx += Math.cos(robot.th)*s; dy += Math.sin(robot.th)*s; }
    }
  if (dx||dy||dth) move(dx, dy, dth);
    else for (const p of particles) { p.x += gauss()*3; p.y += gauss()*3; }
}, 60);
"""

## The page

The app and the single route. Semantic landmark tags carry the content; CSS and the script do the rest.

In [ ]:
app,rt = fast_app(pico=False, hdrs=(Style(css),))

In [ ]:
from fasthtml.svg import Svg, Path

gh = "M12 .297c-6.63 0-12 5.373-12 12 0 5.303 3.438 9.8 8.205 11.385.6.113.82-.258.82-.577 0-.285-.01-1.04-.015-2.04-3.338.724-4.042-1.61-4.042-1.61C4.422 18.07 3.633 17.7 3.633 17.7c-1.087-.744.084-.729.084-.729 1.205.084 1.838 1.236 1.838 1.236 1.07 1.835 2.809 1.305 3.495.998.108-.776.417-1.305.76-1.605-2.665-.305-5.467-1.334-5.467-5.931 0-1.311.469-2.381 1.236-3.221-.124-.303-.535-1.524.117-3.176 0 0 1.008-.322 3.301 1.23A11.509 11.509 0 0112 5.803c1.02.005 2.047.138 3.006.404 2.291-1.552 3.297-1.23 3.297-1.23.653 1.653.242 2.874.118 3.176.77.84 1.235 1.91 1.235 3.221 0 4.609-2.807 5.624-5.479 5.921.43.372.823 1.102.823 2.222 0 1.606-.014 2.898-.014 3.293 0 .322.216.694.825.576C20.565 22.092 24 17.592 24 12.297c0-6.627-5.373-12-12-12"
li = "M20.447 20.452h-3.554v-5.569c0-1.328-.027-3.037-1.852-3.037-1.853 0-2.136 1.445-2.136 2.939v5.667H9.351V9h3.414v1.561h.046c.477-.9 1.637-1.85 3.37-1.85 3.601 0 4.267 2.37 4.267 5.455v6.286zM5.337 7.433a2.062 2.062 0 11.001-4.124 2.062 2.062 0 01-.001 4.124zM7.119 20.452H3.555V9h3.564v11.452zM22.225 0H1.771C.792 0 0 .774 0 1.729v20.542C0 23.227.792 24 1.771 24h20.451C23.2 24 24 23.227 24 22.271V1.729C24 .774 23.2 0 22.225 0z"
xt = "M18.244 2.25h3.308l-7.227 8.26 8.502 11.24H16.17l-5.214-6.817L4.99 21.75H1.68l7.73-8.835L1.254 2.25H8.08l4.713 6.231zm-1.161 17.52h1.833L7.084 4.126H5.117z"
gs = "M5.242 13.769L0 9.5 12 0l12 9.5-5.242 4.269C17.548 11.249 14.978 9.5 12 9.5c-2.977 0-5.548 1.748-6.758 4.269zM12 10a7 7 0 100 14 7 7 0 000-14z"

def Icon(d): return Svg(Path(d=d, fill="currentColor"), viewBox="0 0 24 24", width="22", height="22")
def Soc(name, href, d): return A(Icon(d), href=href, target="_blank", aria_label=name)

In [ ]:
@rt
def index():
    return Title("Lisa Dlima"), Slam_World(Canvas(id="world")), Info_Card(
        Profile_Pic(Img(src="/lisa_pool.jpg", alt="Lisa Dlima")),
        H1("Lisa Dlima"),
        Role_Title("Senior Product Design Engineer @ Google"),
        Bio_Text("Product design engineer with 13 years taking hardware from concept to mass production across robotics, wearables, and data center systems."),
        Exp_List(
            Exp_Item("Google Cloud — data center hardware"),
            Exp_Item("Meta Reality Labs — smart glasses"),
            Exp_Item("Amazon Lab126 — home robotics")),
        Edu_Line("B.S. Mechanical Engineering, The Ohio State University"),
        Pub_Section(
            Pub_Head("Patents"),
            Pub_Item(A("Wheel mechanism for autonomous mobile device — US 11,338,616", href="https://patents.google.com/patent/US11338616", target="_blank")),
            Pub_Item(A("Servicing and maintaining wearable devices", href="https://scholar.google.com/citations?view_op=view_citation&hl=en&user=XzdfIW0AAAAJ&citation_for_view=XzdfIW0AAAAJ:2osOgNQ5qMEC", target="_blank")),
            Pub_Head("Publications"),
            Pub_Item(A("Microvasculature segmentation in HuBMAP — arXiv, 2023", href="https://scholar.google.com/citations?view_op=view_citation&hl=en&user=XzdfIW0AAAAJ&citation_for_view=XzdfIW0AAAAJ:d1gkVwhDpl0C", target="_blank")),
            Pub_Item(A("OSU Underwater Robotics: Riptide AUV — 2016", href="https://scholar.google.com/citations?view_op=view_citation&hl=en&user=XzdfIW0AAAAJ&citation_for_view=XzdfIW0AAAAJ:u5HHmVD_uO8C", target="_blank"))),
        Lesson_Section(
            Lesson_Head("Lessons"),
            Lesson_Item(A("How attention works", href="/attention"))),
        Card_Links(
            Soc("GitHub", "https://github.com/lisadlima", gh),
            Soc("LinkedIn", "https://www.linkedin.com/in/lisadlima/", li),
            Soc("X", "https://x.com/Lisa_Dlima", xt),
            Soc("Scholar", "https://scholar.google.com/citations?user=XzdfIW0AAAAJ", gs))), Hint_Bar("Use arrow keys to drive the robot"), Tilt_Btn("Tap to enable tilt control", onclick="enableTilt();this.remove()"), Script(demo_js)

In [ ]:
@rt('/attention')
def attention(): return FileResponse('attention/attention.html')

## Run

Start the server and view inline.

In [ ]:
from fasthtml.jupyter import *
server = JupyUvi(app)